# Day 2 | Notebook 2: Hybrid Search & Reciprocal Rank Fusion
**RedisVL Focus:** `VectorQuery`, `RangeQuery`, `Tag`/`Num`/`Geo` filters, RRF

---
## 📌 Learning Objectives
1. Understand why pure semantic search fails for structured queries
2. Apply metadata filters (`Tag`, `Num`, `Geo`) to narrow the search space
3. Implement Reciprocal Rank Fusion (RRF) from scratch
4. Build a product recommendation engine using hybrid search
---

In [1]:
# ─── INSTRUCTOR SETTINGS ─────────────────────────────────────────
SHOW_INSIGHTS = False
REDIS_URL = "redis://redis-vector-db:6379"
# ─────────────────────────────────────────────────────────────────
import numpy as np
from redisvl.index import SearchIndex
from redisvl.schema import IndexSchema
from redisvl.query import VectorQuery, RangeQuery
from redisvl.query.filter import Tag, Num
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("✅ Ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Ready


## 📌 Concept: Why Pure Semantic Search Fails

Semantic search finds **meaning** — but real queries often mix meaning with **facts**:
```
Query: 'Sony headphones under $300'

Pure semantic results:
  1. Bose QuietComfort 45 — $329 ← wrong brand, over budget!
  2. Apple AirPods Pro   — $249 ← wrong brand!
  3. Sony WH-1000XM5     — $349 ← right brand, over budget!

Hybrid results (semantic + Tag(brand==Sony) + Num(price<300)):
  1. Sony WH-CH520       — $49  ✓
  2. Sony WH-1000XM4     — $249 ✓
```

**Filters run BEFORE vector search** — they reduce the search space first, then ANN runs on the smaller set.

In [2]:
# Cell 1: Product catalog schema
SCHEMA_DICT = {
    "index": {"name": "nb02_products", "prefix": "prod:", "storage_type": "json"},
    "fields": [
        {"name": "name",        "type": "text"},
        {"name": "brand",       "type": "tag"},
        {"name": "category",    "type": "tag"},
        {"name": "price",       "type": "numeric"},
        {"name": "description", "type": "text"},
        {"name": "embedding",   "type": "vector",
         "attrs": {"dims": 384, "algorithm": "flat", "distance_metric": "cosine"}}
    ]
}
schema = IndexSchema.from_dict(SCHEMA_DICT)
index = SearchIndex(schema, redis_url=REDIS_URL)
index.create(overwrite=True)
print("✅ Product index created")

✅ Product index created


In [3]:
# Cell 2: Load a realistic product catalog (8 products)
CATALOG = [
    {"name": "Aether X1 Flagship",  "brand": "Aether",  "category": "laptop",  "price": 2499.0,
     "description": "Ultra-performance laptop with RTX 4090 and liquid-cooled GPU"},
    {"name": "Aether X1 Pro",        "brand": "Aether",  "category": "laptop",  "price": 1799.0,
     "description": "Professional laptop with RTX 4070 and OLED display"},
    {"name": "Sony WH-1000XM5",      "brand": "Sony",    "category": "audio",   "price": 349.0,
     "description": "Industry-leading noise cancelling wireless headphones"},
    {"name": "Sony WH-1000XM4",      "brand": "Sony",    "category": "audio",   "price": 248.0,
     "description": "Premium wireless headphones with exceptional ANC"},
    {"name": "Sony WH-CH520",        "brand": "Sony",    "category": "audio",   "price": 49.0,
     "description": "Lightweight wireless headphones with 50-hour battery"},
    {"name": "Nebula Tab Pro",        "brand": "Aether",  "category": "tablet",  "price": 899.0,
     "description": "12-inch OLED tablet with M3 chip and Apple Pencil support"},
    {"name": "Bose QC 45",           "brand": "Bose",    "category": "audio",   "price": 329.0,
     "description": "Quiet comfort headphones with world-class noise cancellation"},
    {"name": "Aether USB-C Hub",     "brand": "Aether",  "category": "accessories", "price": 79.0,
     "description": "8-in-1 USB-C hub with 4K HDMI and 100W PD charging"}
]

for p in CATALOG:
    p["embedding"] = model.encode(p["description"]).tolist()

index.load(CATALOG)
print(f"✅ Loaded {len(CATALOG)} products")

✅ Loaded 8 products


## 🔍 Demo 1: Pure Semantic Search — See Its Weakness

In [4]:
# Cell 3: Pure semantic search for 'noise cancelling headphones'
def semantic_search(query_text: str, n: int = 4) -> list:
    q_vec = model.encode(query_text).tolist()
    vq = VectorQuery(
        vector=q_vec,
        vector_field_name="embedding",
        return_fields=["name", "brand", "category", "price"],
        num_results=n
    )
    return index.query(vq)

results = semantic_search("noise cancelling wireless headphones")
print("🔍 Query: 'noise cancelling wireless headphones'")
print(f"{'Rank':>4} {'Name':30} {'Brand':10} {'Price':>8}")
print("-" * 60)
for i, r in enumerate(results, 1):
    print(f"{i:>4} {r['name']:30} {r['brand']:10} ${float(r['price']):>7.2f}")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: Semantic search found all headphones — good!")
    print("   But if user said 'Sony under $300', pure semantic can't enforce that.")
    print("   The filter system runs at the Redis layer, BEFORE the vector search.")

🔍 Query: 'noise cancelling wireless headphones'
Rank Name                           Brand         Price
------------------------------------------------------------
   1 Sony WH-1000XM5                Sony       $ 349.00
   2 Bose QC 45                     Bose       $ 329.00
   3 Sony WH-1000XM4                Sony       $ 248.00
   4 Sony WH-CH520                  Sony       $  49.00


In [5]:
# Cell 4: Add a Tag filter — brand must be 'Sony'
q_vec = model.encode("noise cancelling wireless headphones").tolist()

vq_filtered = VectorQuery(
    vector=q_vec,
    vector_field_name="embedding",
    return_fields=["name", "brand", "price"],
    num_results=4,
    filter_expression=Tag("brand") == "Sony"   # ← filter!
)

filtered_results = index.query(vq_filtered)
print("🔍 Query: 'noise cancelling' WHERE brand == Sony")
for i, r in enumerate(filtered_results, 1):
    print(f"  {i}. {r['name']:30} ${float(r['price']):>7.2f}")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: Filters are applied at Redis FT.SEARCH level.")
    print("   You can inspect the generated query:")
    print(f"   Query string: {str(vq_filtered)}")

🔍 Query: 'noise cancelling' WHERE brand == Sony
  1. Sony WH-1000XM5                $ 349.00
  2. Sony WH-1000XM4                $ 248.00
  3. Sony WH-CH520                  $  49.00


In [6]:
# Cell 5: Compound filter — brand Sony AND price < 300
q_vec = model.encode("noise cancelling wireless headphones").tolist()

brand_filter = Tag("brand") == "Sony"
price_filter = Num("price") < 300
combined_filter = brand_filter & price_filter   # ← AND operator

vq_compound = VectorQuery(
    vector=q_vec,
    vector_field_name="embedding",
    return_fields=["name", "brand", "price"],
    num_results=4,
    filter_expression=combined_filter
)

compound_results = index.query(vq_compound)
print("🔍 Query: 'noise cancelling' WHERE brand==Sony AND price<300")
for i, r in enumerate(compound_results, 1):
    print(f"  {i}. {r['name']:30} ${float(r['price']):>7.2f}")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: '&' chains filters — Redis evaluates them as boolean AND.")
    print("   You can also use '|' for OR and '~' for NOT.")
    print("   Always filter FIRST to reduce search space, then do ANN.")

🔍 Query: 'noise cancelling' WHERE brand==Sony AND price<300
  1. Sony WH-1000XM4                $ 248.00
  2. Sony WH-CH520                  $  49.00


## 📌 Concept: Reciprocal Rank Fusion (RRF)

When you have **two different ranked lists** (e.g., semantic rank + price rank),
RRF combines them into one unified ranking:

```
RRF_score(doc) = Σ_list  1 / (k + rank_in_that_list)

k = 60  (constant that prevents high-rank dominance)

Example:
  doc A: rank 1 in semantic, rank 3 in price
    → 1/(60+1) + 1/(60+3) = 0.01639 + 0.01587 = 0.03226

  doc B: rank 2 in semantic, rank 1 in price
    → 1/(60+2) + 1/(60+1) = 0.01613 + 0.01639 = 0.03252

  doc B wins (strong in BOTH lists, not just one)
```

In [7]:
# Cell 6: Implement RRF from scratch
def reciprocal_rank_fusion(lists_of_ids: list[list[str]], k: int = 60) -> list[tuple]:
    """
    Merge multiple ranked lists using RRF.
    
    Args:
        lists_of_ids: list of ranked lists, each containing doc IDs
        k: smoothing constant (default 60)
    Returns:
        Sorted list of (doc_id, rrf_score) tuples, highest first
    """
    scores = {}
    for ranked_list in lists_of_ids:
        for rank, doc_id in enumerate(ranked_list, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

# Test with simple example
semantic_rank = ["sony_xm5", "bose_qc45", "sony_xm4", "sony_ch520"]
price_rank    = ["sony_ch520", "sony_xm4", "bose_qc45", "sony_xm5"]  # cheapest first

merged = reciprocal_rank_fusion([semantic_rank, price_rank])
print("📊 RRF Merged Ranking:")
for rank, (doc_id, score) in enumerate(merged, 1):
    sem_pos = semantic_rank.index(doc_id) + 1 if doc_id in semantic_rank else 'N/A'
    price_pos = price_rank.index(doc_id) + 1 if doc_id in price_rank else 'N/A'
    print(f"  {rank}. {doc_id:15} score={score:.5f}  (sem_rank={sem_pos}, price_rank={price_pos})")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: sony_xm4 is ranked 3rd in semantic but 2nd in price.")
    print("   Consistent mid-rank in BOTH lists gives it a high RRF score.")
    print("   This is 'consensus ranking' — docs good at multiple signals win.")

📊 RRF Merged Ranking:
  1. sony_xm5        score=0.03202  (sem_rank=1, price_rank=4)
  2. sony_ch520      score=0.03202  (sem_rank=4, price_rank=1)
  3. bose_qc45       score=0.03200  (sem_rank=2, price_rank=3)
  4. sony_xm4        score=0.03200  (sem_rank=3, price_rank=2)


In [8]:
# Cell 7: Real hybrid — semantic + price RRF on our product index
def hybrid_search_with_rrf(query_text: str, max_price: float = 9999.0, top_n: int = 4):
    q_vec = model.encode(query_text).tolist()

    # List 1: Semantic ranking
    vq = VectorQuery(q_vec, "embedding",
                     return_fields=["name", "price"],
                     filter_expression=Num("price") <= max_price,
                     num_results=8)
    sem_results = index.query(vq)
    sem_ids = [r["name"] for r in sem_results]

    # List 2: Price ranking (cheapest first = best value)
    price_ids = [r["name"] for r in sorted(sem_results, key=lambda x: float(x["price"]))]

    # Merge with RRF
    merged = reciprocal_rank_fusion([sem_ids, price_ids])

    # Attach prices for display
    price_lookup = {r["name"]: float(r["price"]) for r in sem_results}
    return [(name, score, price_lookup.get(name, 0)) for name, score in merged[:top_n]]

results = hybrid_search_with_rrf("wireless headphones", max_price=300)
print("🔍 Hybrid Search: 'wireless headphones' under $300")
print(f"  {'Rank':4} {'Name':30} {'RRF Score':12} {'Price':>8}")
print("-" * 60)
for i, (name, score, price) in enumerate(results, 1):
    print(f"  {i:<4} {name:30} {score:.5f}      ${price:>7.2f}")

🔍 Hybrid Search: 'wireless headphones' under $300
  Rank Name                           RRF Score       Price
------------------------------------------------------------
  1    Sony WH-CH520                  0.03279      $  49.00
  2    Sony WH-1000XM4                0.03200      $ 248.00
  3    Aether USB-C Hub               0.03200      $  79.00


In [9]:
# Cell 8: Edge case — exact serial number (semantic search fails alone)
# Semantic: embeds 'WH-1000XM5' → finds audio products but may miss exact model
# Keyword filter: exactly matches the name text field

from redisvl.query.filter import Text

# Simulate serial number search
q_vec = model.encode("WH-1000XM5").tolist()

# Without text filter
vq_no_filter = VectorQuery(q_vec, "embedding", return_fields=["name", "price"], num_results=3)
no_filter_results = index.query(vq_no_filter)

# With text filter on name (keyword match)
vq_text = VectorQuery(q_vec, "embedding", return_fields=["name", "price"],
                      filter_expression=Text("name") % "*XM5*",  # wildcard match
                      num_results=3)
text_results = index.query(vq_text)

print("🔍 Searching for serial: 'WH-1000XM5'")
print("\nWithout text filter (pure semantic):")
for r in no_filter_results: print(f"  {r['name']}")
print("\nWith text filter (*XM5* wildcard):")
for r in text_results: print(f"  {r['name']}")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: Text % pattern uses Redis full-text search (FT.SEARCH).")
    print("   '*XM5*' = contains 'XM5'. This is O(1) in Redis's inverted index.")
    print("   Production tip: combine both signals for max robustness.")

🔍 Searching for serial: 'WH-1000XM5'

Without text filter (pure semantic):
  Aether X1 Flagship
  Nebula Tab Pro
  Aether USB-C Hub

With text filter (*XM5* wildcard):
  Sony WH-1000XM5


## ✅ Checkpoint

In [10]:
# Cell 9: Checkpoint
score = 0

# Test 1: filters worked — all results within budget
if all(float(r['price']) < 300 for r in compound_results):
    print("✅ Test 1 PASS: All compound-filter results are under $300")
    score += 1
else:
    print("❌ Test 1 FAIL: Found results over $300 — filter not applied correctly")

# Test 2: RRF returns correct type
rrf_test = reciprocal_rank_fusion([["a", "b", "c"], ["c", "a", "b"]])
if rrf_test[0][0] in ["a", "c"]:
    print("✅ Test 2 PASS: RRF returns valid merged ranking")
    score += 1
else:
    print("❌ Test 2 FAIL: RRF output format incorrect")

# Test 3: hybrid returned results under budget
if all(price <= 300 for _, _, price in results):
    print("✅ Test 3 PASS: Hybrid search respects price filter")
    score += 1
else:
    print("❌ Test 3 FAIL: Hybrid search returned results over budget")

print(f"\n🏆 Score: {score}/3")

✅ Test 1 PASS: All compound-filter results are under $300
✅ Test 2 PASS: RRF returns valid merged ranking
✅ Test 3 PASS: Hybrid search respects price filter

🏆 Score: 3/3


---
## 📝 Assignment: Product Recommendation Engine

**Scenario**: Build a recommendation engine that takes a structured user preference object
and returns the best-matching products using hybrid search + RRF.

**Task 1**: Implement `build_user_query()` — build a `VectorQuery` with multi-field filters.

**Task 2**: Implement `multi_signal_rrf()` — merge semantic, price, and brand-preference rankings.

**Task 3 (Bonus)**: Add a `RangeQuery` to find all products within a price band `[low, high]`.

In [11]:
# ── ASSIGNMENT ────────────────────────────────────────────────────────────────
from redisvl.query import RangeQuery

USER_PREFERENCE = {
    "semantic_query": "noise cancelling wireless audio",
    "preferred_brands": ["Sony", "Bose"],
    "max_budget": 400,
    "preferred_categories": ["audio"]
}

def build_user_query(preference: dict) -> VectorQuery:
    """
    TODO: Build a VectorQuery that:
    1. Embeds preference['semantic_query']
    2. Filters: category in preferred_categories
    3. Filters: price <= max_budget
    4. Returns: name, brand, category, price fields
    5. Returns top 6 results

    Hint: Tag("category") == "audio" uses a single value.
          For multiple brands, use: Tag("brand") == "Sony" | Tag("brand") == "Bose"
    """
    # TODO
    pass


def multi_signal_rrf(query_results: list, preference: dict, k: int = 60) -> list:
    """
    TODO: Merge 3 ranking signals using RRF:
    Signal 1: semantic relevance (order from query_results)
    Signal 2: price (ascending — cheapest = best value)
    Signal 3: brand preference (preferred brands ranked first)

    Return: list of (name, rrf_score, price, brand) tuples, sorted by rrf_score desc
    """
    # TODO
    pass


# BONUS Task 3: RangeQuery
def find_products_in_price_range(query_text: str, low: float, high: float):
    """
    TODO: Use RangeQuery to find products where the semantic distance
    is within [0, 0.4] AND price is between [low, high].

    RangeQuery signature:
        RangeQuery(vector, vector_field_name, distance_threshold,
                   return_fields, filter_expression)
    """
    # TODO
    pass


# Uncomment to test your implementation:
# q = build_user_query(USER_PREFERENCE)
# results = index.query(q)
# final = multi_signal_rrf(results, USER_PREFERENCE)
# for rank, (name, score, price, brand) in enumerate(final, 1):
#     print(f"{rank}. {name:30} ${price:>7.2f} [{brand}] score={score:.5f}")

print("📝 Assignment ready — implement the TODO functions above")

📝 Assignment ready — implement the TODO functions above
